In [26]:
import pandas as pd
import os

In [27]:
dataframes = {
    "distribution_centers": pd.read_parquet(
        "../cleaned_data/distribution_centers.parquet"
    ),
    "products": pd.read_parquet("../cleaned_data/products.parquet"),
    "inventory_items": pd.read_parquet("../cleaned_data/inventory_items.parquet"),
    "users": pd.read_parquet("../cleaned_data/users.parquet"),
    "orders": pd.read_parquet("../cleaned_data/orders.parquet"),
    "order_items": pd.read_parquet("../cleaned_data/order_items.parquet"),
    "events": pd.read_parquet("../cleaned_data/events.parquet"),
}


In [28]:
expected_schema = {
    "distribution_centers": ["id", "name"],
    "products": [
        "id",
        "cost",
        "category",
        "name",
        "brand",
        "retail_price",
        "department",
        "distribution_center_id",
    ],
    "inventory_items": [
        "id",
        "product_id",
        "created_at",
        "sold_at",
        "cost",
        "product_category",
        "product_name",
        "product_brand",
        "product_retail_price",
        "product_department",
        "product_distribution_center_id",
    ],
    "users": [
        "id",
        "full_name",
        "email",
        "age",
        "gender",
        "state",
        "street_address",
        "postal_code",
        "city",
        "country",
        "traffic_source",
        "created_at",
    ],
    "orders": [
        "order_id",
        "user_id",
        "status",
        "created_at",
        "returned_at",
        "shipped_at",
        "delivered_at",
        "num_of_item",
    ],
    "order_items": [
        "id",
        "order_id",
        "user_id",
        "product_id",
        "inventory_item_id",
        "status",
        "sale_price",
    ],
    "events": [
        "id",
        "user_id",
        "sequence_number",
        "session_id",
        "created_at",
        "ip_address",
        "city",
        "state",
        "postal_code",
        "browser",
        "traffic_source",
        "uri",
        "event_type",
        "user_type",
    ],
}


In [29]:
schema_report = []

for table_name, df in dataframes.items():
    expected_columns = set(expected_schema[table_name])
    actual_columns = set(df.columns)

    missing_columns = expected_columns - actual_columns
    extra_columns = actual_columns - expected_columns

    schema_report.append(
        {
            "table": table_name,
            "total_columns": len(df.columns),
            "missing_columns": list(missing_columns),
            "extra_columns": list(extra_columns),
            "schema_valid": len(missing_columns) == 0 and len(extra_columns) == 0,
        }
    )

schema_report_df = pd.DataFrame(schema_report)

schema_report_df


,table,total_columns,missing_columns,extra_columns,schema_valid
0,distribution_centers,2,[],[],True
1,products,8,[],[],True
2,inventory_items,12,[],[available],False
3,users,12,[],[],True
4,orders,8,[],[],True
5,order_items,7,[],[],True
6,events,14,[],[],True


In [30]:
for table_name, df in dataframes.items():
    print(f"\n{'=' * 50}")
    print(f"TABLE: {table_name}")
    print("=" * 50)

    print(df.dtypes)



TABLE: distribution_centers
id      string
name       str
dtype: object

TABLE: products
id                         string
cost                      float64
category                   string
name                       string
brand                      string
retail_price              float64
department                 string
distribution_center_id     string
dtype: object

TABLE: inventory_items
id                                             string
product_id                                     string
created_at                        datetime64[us, UTC]
sold_at                           datetime64[us, UTC]
cost                                          float64
product_category                                  str
product_name                                      str
product_brand                                     str
product_retail_price                          float64
product_department                                str
product_distribution_center_id                 string
availa

In [31]:
primary_keys = {
    "distribution_centers": "id",
    "products": "id",
    "inventory_items": "id",
    "users": "id",
    "orders": "order_id",
    "order_items": "id",
    "events": "id",
}


In [32]:
pk_report = []

for table_name, pk_column in primary_keys.items():
    df = dataframes[table_name]

    null_count = df[pk_column].isna().sum()
    duplicate_count = df[pk_column].duplicated().sum()

    pk_report.append(
        {
            "table": table_name,
            "primary_key": pk_column,
            "null_count": null_count,
            "duplicate_count": duplicate_count,
            "pk_valid": null_count == 0 and duplicate_count == 0,
        }
    )

pk_report_df = pd.DataFrame(pk_report)

pk_report_df


,table,primary_key,null_count,duplicate_count,pk_valid
0,distribution_centers,id,0,0,True
1,products,id,0,26,False
2,inventory_items,id,0,0,True
3,users,id,0,0,True
4,orders,order_id,0,0,True
5,order_items,id,0,0,True
6,events,id,0,0,True


In [33]:
foreign_keys = [
    {
        "child_table": "orders",
        "child_column": "user_id",
        "parent_table": "users",
        "parent_column": "id",
    },
    {
        "child_table": "order_items",
        "child_column": "order_id",
        "parent_table": "orders",
        "parent_column": "order_id",
    },
    {
        "child_table": "order_items",
        "child_column": "user_id",
        "parent_table": "users",
        "parent_column": "id",
    },
    {
        "child_table": "order_items",
        "child_column": "product_id",
        "parent_table": "products",
        "parent_column": "id",
    },
    {
        "child_table": "order_items",
        "child_column": "inventory_item_id",
        "parent_table": "inventory_items",
        "parent_column": "id",
    },
    {
        "child_table": "inventory_items",
        "child_column": "product_id",
        "parent_table": "products",
        "parent_column": "id",
    },
    {
        "child_table": "products",
        "child_column": "distribution_center_id",
        "parent_table": "distribution_centers",
        "parent_column": "id",
    },
    {
        "child_table": "inventory_items",
        "child_column": "product_distribution_center_id",
        "parent_table": "distribution_centers",
        "parent_column": "id",
    },
    {
        "child_table": "events",
        "child_column": "user_id",
        "parent_table": "users",
        "parent_column": "id",
    },
]


In [34]:
fk_report = []

for relation in foreign_keys:
    child_table = relation["child_table"]
    child_column = relation["child_column"]

    parent_table = relation["parent_table"]
    parent_column = relation["parent_column"]

    child_df = dataframes[child_table]
    parent_df = dataframes[parent_table]

    child_values = child_df[child_column].dropna()

    parent_values = parent_df[parent_column].dropna()

    child_values = child_values.astype("string")
    parent_values = parent_values.astype("string")

    orphan_count = (~child_values.isin(parent_values)).sum()

    fk_report.append(
        {
            "relationship": f"{child_table}.{child_column} -> {parent_table}.{parent_column}",
            "orphan_count": orphan_count,
            "fk_valid": orphan_count == 0,
        }
    )

fk_report_df = pd.DataFrame(fk_report)

fk_report_df


,relationship,orphan_count,fk_valid
0,orders.user_id -> users.id,0,True
1,order_items.order_id -> orders.order_id,0,True
2,order_items.user_id -> users.id,0,True
3,order_items.product_id -> products.id,0,True
4,order_items.inventory_item_id -> inventory_ite...,0,True
5,inventory_items.product_id -> products.id,0,True
6,products.distribution_center_id -> distributio...,0,True
7,inventory_items.product_distribution_center_id...,0,True
8,events.user_id -> users.id,0,True


In [35]:
missing_report = []

for table_name, df in dataframes.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        missing_percentage = (missing_count / len(df)) * 100

        missing_report.append(
            {
                "table": table_name,
                "column": column,
                "missing_count": missing_count,
                "missing_percentage": round(missing_percentage, 2),
            }
        )

missing_report_df = pd.DataFrame(missing_report)

missing_report_df = missing_report_df.sort_values(
    by="missing_percentage", ascending=False
)

missing_report_df


,table,column,missing_count,missing_percentage
38,orders,returned_at,112353,89.94
40,orders,delivered_at,81325,65.10
13,inventory_items,sold_at,306949,62.91
50,events,user_id,1124065,46.43
39,orders,shipped_at,43671,34.96
...,...,...,...,...
58,events,browser,0,0.00
59,events,traffic_source,0,0.00
60,events,uri,0,0.00
61,events,event_type,0,0.00


In [36]:
os.makedirs("../validation/reports", exist_ok=True)

schema_report_df.to_csv("../validation/reports/schema_report.csv", index=False)

pk_report_df.to_csv("../validation/reports/primary_key_report.csv", index=False)

fk_report_df.to_csv("../validation/reports/foreign_key_report.csv", index=False)

missing_report_df.to_csv("../validation/reports/missing_values_report.csv", index=False)
